# Synthetic DIHS Scenarios

This notebook generates synthetic two-dimensional datasets designed to illustrate how DIHS behaves under different geometric relationships between source classes and unknown samples.

## 1. Imports and Notebook Configuration
        

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, List

# Global style
plt.rcParams.update({
    "font.family": "serif",
    "svg.fonttype": "none",
    "figure.dpi": 350,
    "savefig.dpi": 350,
})

RANDOM_SEED = 1235
N_SOURCE_DEFAULT = 120
N_UNKNOWN_DEFAULT = 20

REPO_ROOT = Path("..").resolve()
OUTDIR = REPO_ROOT / "data" / "processed" / "synthetic_scenarios"
OUTDIR.mkdir(parents=True, exist_ok=True)


NameError: name '__file__' is not defined

## 2. Sampling Helpers

Define the Gaussian specification container and the helper functions used throughout the scenario builders.
        

In [ ]:
@dataclass
class GaussianSpec:
    mean: Tuple[float, float]
    cov: np.ndarray
    n: int
    class_label: str

def sample_gaussian(rng: np.random.Generator, spec: GaussianSpec) -> pd.DataFrame:
    pts = rng.multivariate_normal(mean=spec.mean, cov=spec.cov, size=spec.n)
    return pd.DataFrame({"class_label": spec.class_label, "x1": pts[:, 0], "x2": pts[:, 1]})

def sample_mixture(rng: np.random.Generator, components: List[GaussianSpec], final_label: str) -> pd.DataFrame:
    parts = [sample_gaussian(rng, spec) for spec in components]
    df = pd.concat(parts, ignore_index=True)
    df["class_label"] = final_label
    return df

def rotate_cov(var_major: float, var_minor: float, angle_deg: float) -> np.ndarray:
    theta = np.deg2rad(angle_deg)
    c, s = np.cos(theta), np.sin(theta)
    r = np.array([[c, -s], [s, c]])
    d = np.diag([var_major, var_minor])
    return r @ d @ r.T

def add_metadata(df: pd.DataFrame, scenario: str, role: str, true_source: str | None) -> pd.DataFrame:
    out = df.copy()
    out["scenario"] = scenario
    out["role"] = role
    if role == "source":
        out["is_true_source"] = out["class_label"].eq(true_source)
    else:
        out["is_true_source"] = True if true_source is not None else pd.NA
    return out[["scenario", "role", "class_label", "x1", "x2", "is_true_source"]]


## 3. Scenario Builders

In [ ]:
# Scenario builders
def scenario_compact_separated(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"
    specs = [
        GaussianSpec(mean=(-4.0, 2.0), cov=np.array([[0.35, 0.0], [0.0, 0.25]]), n=n_source, class_label="A"),
        GaussianSpec(mean=(3.5, 3.5), cov=np.array([[0.45, 0.0], [0.0, 0.30]]), n=n_source, class_label="B"),
        GaussianSpec(mean=(0.5, -3.5), cov=np.array([[0.40, 0.0], [0.0, 0.35]]), n=n_source, class_label="C"),
    ]
    sources = pd.concat([sample_gaussian(rng, s) for s in specs], ignore_index=True)
    unknown = sample_gaussian(rng, GaussianSpec(mean=(-4.0, 2.0), cov=np.array([[0.20, 0.0], [0.0, 0.15]]), n=n_unknown, class_label="X"))
    return pd.concat([add_metadata(sources, "compact_separated", "source", true_source),
                      add_metadata(unknown, "compact_separated", "unknown", true_source)], ignore_index=True), true_source

def scenario_one_sided_embedding(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"
    specs = [
        GaussianSpec(mean=(0.0, 0.0), cov=np.array([[0.18, 0.0], [0.0, 0.14]]), n=max(60, n_source // 2), class_label="A"),
        GaussianSpec(mean=(0.0, 0.0), cov=np.array([[2.5, 0.6], [0.6, 1.9]]), n=n_source * 2, class_label="B"),
        GaussianSpec(mean=(5.5, 3.5), cov=np.array([[0.5, 0.0], [0.0, 0.4]]), n=n_source, class_label="C"),
    ]
    sources = pd.concat([sample_gaussian(rng, s) for s in specs], ignore_index=True)
    unknown = sample_gaussian(rng, GaussianSpec(mean=(0.1, -0.1), cov=np.array([[0.12, 0.0], [0.0, 0.10]]), n=n_unknown, class_label="X"))
    return pd.concat([add_metadata(sources, "one_sided_embedding", "source", true_source),
                      add_metadata(unknown, "one_sided_embedding", "unknown", true_source)], ignore_index=True), true_source

def scenario_shallow_only_overlap(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"
    specs = [
        GaussianSpec(mean=(-0.8, 0.2), cov=rotate_cov(2.0, 0.12, 35), n=n_source, class_label="A"),
        GaussianSpec(mean=(0.8, -0.2), cov=rotate_cov(2.0, 0.12, -35), n=n_source, class_label="B"),
        GaussianSpec(mean=(4.5, 3.5), cov=np.array([[0.5, 0.0], [0.0, 0.5]]), n=n_source, class_label="C"),
    ]
    sources = pd.concat([sample_gaussian(rng, s) for s in specs], ignore_index=True)
    unknown = sample_gaussian(rng, GaussianSpec(mean=(-1.4, 0.7), cov=np.array([[0.20, 0.04], [0.04, 0.18]]), n=n_unknown, class_label="X"))
    return pd.concat([add_metadata(sources, "shallow_only_overlap", "source", true_source),
                      add_metadata(unknown, "shallow_only_overlap", "unknown", true_source)], ignore_index=True), true_source

def scenario_persistent_overlap(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"
    specs = [
        GaussianSpec(mean=(0.0, 0.0), cov=rotate_cov(1.0, 0.25, 20), n=n_source, class_label="A"),
        GaussianSpec(mean=(0.4, 0.2), cov=rotate_cov(1.2, 0.30, 20), n=n_source, class_label="B"),
        GaussianSpec(mean=(5.0, -3.0), cov=np.array([[0.5, 0.0], [0.0, 0.45]]), n=n_source, class_label="C"),
    ]
    sources = pd.concat([sample_gaussian(rng, s) for s in specs], ignore_index=True)
    unknown = sample_gaussian(rng, GaussianSpec(mean=(0.15, 0.1), cov=np.array([[0.22, 0.03], [0.03, 0.18]]), n=n_unknown, class_label="X"))
    return pd.concat([add_metadata(sources, "persistent_overlap", "source", true_source),
                      add_metadata(unknown, "persistent_overlap", "unknown", true_source)], ignore_index=True), true_source

def scenario_multimodal_class(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"

    a1 = GaussianSpec(
        mean=(-1.6, -1.45),
        cov=np.array([[0.34, 0.04], [0.04, 0.26]]),
        n=n_source // 2,
        class_label="A",
    )
    a2 = GaussianSpec(
        mean=(1.55, 1.45),
        cov=np.array([[0.34, 0.03], [0.03, 0.26]]),
        n=n_source // 2,
        class_label="A",
    )
    A = sample_mixture(rng, [a1, a2], final_label="A")

    specs_other = [
        GaussianSpec(
            mean=(0.0, 0.0),
            cov=np.array([[0.42, 0.08], [0.08, 0.38]]),
            n=n_source,
            class_label="B",
        ),

        GaussianSpec(
            mean=(3.4, -2.2),
            cov=np.array([[0.50, 0.0], [0.0, 0.35]]),
            n=n_source,
            class_label="C",
        ),
    ]

    others = pd.concat([sample_gaussian(rng, s) for s in specs_other], ignore_index=True)
    sources = pd.concat([A, others], ignore_index=True)

    n_x1 = int(0.60 * n_unknown)
    n_x2 = n_unknown - n_x1

    unknown = sample_mixture(
        rng,
        [
            GaussianSpec(
                mean=(-1.55, -1.40),
                cov=np.array([[0.20, 0.03], [0.03, 0.15]]),
                n=n_x1,
                class_label="X",
            ),
            GaussianSpec(
                mean=(1.50, 1.38),
                cov=np.array([[0.20, 0.02], [0.02, 0.15]]),
                n=n_x2,
                class_label="X",
            ),
        ],
        final_label="X",
    )

    return pd.concat(
        [
            add_metadata(sources, "multimodal_class", "source", true_source),
            add_metadata(unknown, "multimodal_class", "unknown", true_source),
        ],
        ignore_index=True,
    ), true_source

def scenario_internal_heterogeneity(rng, n_source=N_SOURCE_DEFAULT, n_unknown=N_UNKNOWN_DEFAULT):
    true_source = "A"

    A = sample_mixture(
        rng,
        [
            GaussianSpec(
                mean=(-2.3, -0.9),
                cov=rotate_cov(1.55, 0.30, 28),
                n=n_source // 2,
                class_label="A",
            ),
            GaussianSpec(
                mean=(-0.4, 0.9),
                cov=rotate_cov(1.35, 0.32, -18),
                n=n_source // 2,
                class_label="A",
            ),
        ],
        final_label="A",
    )

    specs_other = [
        GaussianSpec(
            mean=(0.95, 1.35),
            cov=rotate_cov(0.95, 0.35, -10),
            n=n_source,
            class_label="B",
        ),

        GaussianSpec(
            mean=(3.9, -2.0),
            cov=np.array([[0.55, 0.05], [0.05, 0.45]]),
            n=n_source,
            class_label="C",
        ),
    ]

    others = pd.concat([sample_gaussian(rng, s) for s in specs_other], ignore_index=True)
    sources = pd.concat([A, others], ignore_index=True)

    n_x1 = int(0.35 * n_unknown)
    n_x2 = int(0.35 * n_unknown)
    n_x3 = n_unknown - n_x1 - n_x2

    unknown = sample_mixture(
        rng,
        [
            GaussianSpec(
                mean=(-2.05, -0.75),
                cov=rotate_cov(0.35, 0.09, 25),
                n=n_x1,
                class_label="X",
            ),
            GaussianSpec(
                mean=(-1.25, 0.00),
                cov=rotate_cov(0.38, 0.10, 15),
                n=n_x2,
                class_label="X",
            ),
            GaussianSpec(
                mean=(-0.55, 0.75),
                cov=rotate_cov(0.35, 0.09, -15),
                n=n_x3,
                class_label="X",
            ),
        ],
        final_label="X",
    )

    return pd.concat(
        [
            add_metadata(sources, "internal_heterogeneity", "source", true_source),
            add_metadata(unknown, "internal_heterogeneity", "unknown", true_source),
        ],
        ignore_index=True,
    ), true_source


## 4. Generate Scenarios
        

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

builders = {
    "compact_separated": scenario_compact_separated,
    "one_sided_embedding": scenario_one_sided_embedding,
    "shallow_only_overlap": scenario_shallow_only_overlap,
    "persistent_overlap": scenario_persistent_overlap,
    "multimodal_class": scenario_multimodal_class,
    "internal_heterogeneity": scenario_internal_heterogeneity,
}

scenario_dfs = {}
all_frames = []

for name, builder in builders.items():
    df, _ = builder(rng)
    scenario_dfs[name] = df
    all_frames.append(df)

combined = pd.concat(all_frames, ignore_index=True)
combined_path = OUTDIR / "all_scenarios_combined.csv"
combined.to_csv(combined_path, index=False)


## 5. Create Comparison Figure

In [ ]:
pad = 0.75
xmin, xmax = combined["x1"].min() - pad, combined["x1"].max() + pad
ymin, ymax = combined["x2"].min() - pad, combined["x2"].max() + pad

fig, axes = plt.subplots(2, 4, figsize=(18, 9.5), constrained_layout=True)
axes = axes.flatten()

scenario_items = list(scenario_dfs.items())
panel_labels = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

for i, (ax, (name, df)) in enumerate(zip(axes[:6], scenario_items)):
    source_df = df[df["role"] == "source"]
    unknown_df = df[df["role"] == "unknown"]

    for cls in sorted(source_df["class_label"].unique()):
        sub = source_df[source_df["class_label"] == cls]
        ax.scatter(
            sub["x1"],
            sub["x2"],
            s=34,
            alpha=0.78,
            label=cls,
            zorder=3,
        )

    ax.scatter(
        unknown_df["x1"],
        unknown_df["x2"],
        s=55,
        c="black",
        alpha=0.99,
        marker="x",
        linewidths=2.4,
        label="X",
        zorder=4,
    )

    title = name.replace("_", " ")
    ax.set_title(title.capitalize(), fontsize=22, pad=10)

    ax.set_axisbelow(True)
    ax.grid(True, color="0.85", linestyle="--", linewidth=0.8, zorder=-1)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_box_aspect(1)

    ax.set_xlabel("x₁", fontsize=26, labelpad=2)
    ax.set_ylabel("x₂", fontsize=26, labelpad=2)

    ax.tick_params(axis="both", labelsize=16, width=1.2, length=6)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    ax.text(
        0.5,
        -0.30,
        panel_labels[i],
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontweight="bold",
        fontsize=22,
    )

# 7th panel for legend
legend_ax = axes[6]
legend_ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
legend = legend_ax.legend(
    handles,
    labels,
    loc="center",
    title="Classes",
    title_fontsize=22,
    fontsize=19,
    ncol=1,
    frameon=True,
    fancybox=False,
    borderpad=1.2,
    labelspacing=1.0,
    handlelength=1.6,
    handletextpad=0.8,
    borderaxespad=0.0,
    markerscale=1.3,
)

legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("0.75")
legend.get_frame().set_linewidth(1.2)

# 8th panel empty
axes[7].axis("off")

svg_path = OUTDIR / "synthetic_scenarios.svg"

fig.savefig(svg_path, format="svg", bbox_inches="tight")
plt.close(fig)

print(f"Saved: {svg_path}")
print(f"Saved: {combined_path}")

Saved: C:\Users\jayme\Desktop\14_07_2026_PreZenodo\DIHS_Correlator\data\Processed\synthetic_scenarios\synthetic_scenarios.svg
Saved: C:\Users\jayme\Desktop\14_07_2026_PreZenodo\DIHS_Correlator\data\Processed\synthetic_scenarios\all_scenarios_combined.csv
